### Data Cleaning Dataset Lomba

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load dataset
df = pd.read_csv("dataset_mentah_kotor.csv")

print("dataset_mentah_kotor.csv:")
display(df.head())

dataset_mentah_kotor.csv:


,nama_lomba,bidang_utama,kategori,tema,jumlah_peserta,hadiah,start_date,deadline_date,penyelenggara
0,National DevFest 2025,Tech & IT,UI/UX Design,Smart City,2371,"IDR 45,000,000",2025-04-25,"Jul 18, 2025",NaN
1,Festival Eco Summit Samarinda,Sosial & Lingkungan,Kemanusiaan,Aksi Tanggap Bencana,2660,Rp 19.000.000,"May 02, 2024","Jul 01, 2024",WWF Indonesia
2,Festival Health Challenge Malang,Kesehatan,Gizi dan Nutrisi,Pencegahan Stunting,2771,Rp 18.000.000,"Mar 27, 2026","May 27, 2026",Kementerian Kesehatan RI
3,National Ideathon 2024,Tech & IT,Software Development,Blockchain for Good,3110 orang,Rp 12.000.000,"Feb 05, 2024","Apr 19, 2024",Kementerian Kominfo
4,GELAR DEVFEST BALI,Tech & IT,Software Development,Blockchain for Good,1592,Rp 18.000.000,"Sep 26, 2025","Nov 05, 2025",aws indonesia


In [3]:
print("========= ASSESSING df =========")
df.info()
print("\nMissing values di df:\n", df.isna().sum())
print("\nJumlah duplikasi di df:", df.duplicated().sum())
print("\nStatistik deskriptif df:\n", df.describe())

========= ASSESSING df =========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   nama_lomba      1003 non-null   object
 1   bidang_utama    1017 non-null   object
 2   kategori        1003 non-null   object
 3   tema            1008 non-null   object
 4   jumlah_peserta  1003 non-null   object
 5   hadiah          1007 non-null   object
 6   start_date      1022 non-null   object
 7   deadline_date   1001 non-null   object
 8   penyelenggara   1023 non-null   object
dtypes: object(9)
memory usage: 77.5+ KB

Missing values di df:
 nama_lomba        97
bidang_utama      83
kategori          97
tema              92
jumlah_peserta    97
hadiah            93
start_date        78
deadline_date     99
penyelenggara     77
dtype: int64

Jumlah duplikasi di df: 1

Statistik deskriptif df:
                              nama_lomba      bidang_utam

nama lomba : object
bidang_utama : categorical
kategori : categorical
tema : categorical
jumlah_peserta : int64
hadiah : int64
start_date : datetime64
deadline_date : datetime64
penyelenggara : object

In [4]:
df = df.drop_duplicates()
print("\nJumlah duplikasi setelah dihapus:", df.duplicated().sum())


Jumlah duplikasi setelah dihapus: 0


In [5]:
# Hapus baris jika kolom 'nama_lomba' bernilai NaN
df.dropna(subset=['nama_lomba'], inplace=True)
print(f"Total baris setelah drop nama_lomba kosong: {len(df)}")

Total baris setelah drop nama_lomba kosong: 1002


In [6]:
# A. Membersihkan kolom 'hadiah' (hapus Rp, IDR, titik, koma, spasi)
# Regex r'[^\d]' artinya hapus semua karakter yang BUKAN angka
df['hadiah'] = df['hadiah'].astype(str).str.replace(r'[^\d]', '', regex=True)
df.head()

,nama_lomba,bidang_utama,kategori,tema,jumlah_peserta,hadiah,start_date,deadline_date,penyelenggara
0,National DevFest 2025,Tech & IT,UI/UX Design,Smart City,2371,45000000,2025-04-25,"Jul 18, 2025",NaN
1,Festival Eco Summit Samarinda,Sosial & Lingkungan,Kemanusiaan,Aksi Tanggap Bencana,2660,19000000,"May 02, 2024","Jul 01, 2024",WWF Indonesia
2,Festival Health Challenge Malang,Kesehatan,Gizi dan Nutrisi,Pencegahan Stunting,2771,18000000,"Mar 27, 2026","May 27, 2026",Kementerian Kesehatan RI
3,National Ideathon 2024,Tech & IT,Software Development,Blockchain for Good,3110 orang,12000000,"Feb 05, 2024","Apr 19, 2024",Kementerian Kominfo
4,GELAR DEVFEST BALI,Tech & IT,Software Development,Blockchain for Good,1592,18000000,"Sep 26, 2025","Nov 05, 2025",aws indonesia


In [7]:
print("\nMissing values di df:\n", df.isna().sum())


Missing values di df:
 nama_lomba         0
bidang_utama      79
kategori          90
tema              88
jumlah_peserta    85
hadiah             0
start_date        70
deadline_date     97
penyelenggara     70
dtype: int64


In [8]:
import pandas as pd
import numpy as np

# Fungsi detektif untuk menebak bidang berdasarkan judul lomba
def tebak_bidang_dari_nama(nama):
    nama = str(nama).lower() # Jadikan huruf kecil semua biar gampang dicari
    
    # Cek kata kunci satu per satu
    if any(kata in nama for kata in ['med', 'sehat', 'health']):
        return 'Kesehatan'
    elif any(kata in nama for kata in ['tech', 'hackathon', 'data', 'code', 'dev']):
        return 'Tech & IT'
    elif any(kata in nama for kata in ['bisnis', 'preneur', 'startup', 'plan', 'fintech']):
        return 'Bisnis & Ekonomi'
    elif any(kata in nama for kata in ['sosial', 'eco', 'lingkungan', 'relawan', 'impact']):
        return 'Sosial & Lingkungan'
    else:
        return 'Lainnya' # Kalau judulnya bener-bener nggak bisa ditebak

# ==========================================
# EKSEKUSI IMPUTASI CERDAS
# ==========================================

# 1. Cari baris mana saja yang 'bidang_utama'-nya masih kosong (NaN)
mask_kosong = df['bidang_utama'].isna() 

# 2. Terapkan fungsi detektif HANYA pada baris yang kosong tersebut
df.loc[mask_kosong, 'bidang_utama'] = df.loc[mask_kosong, 'nama_lomba'].apply(tebak_bidang_dari_nama)

# (Lakukan logika yang sama untuk 'kategori' atau 'tema' jika mau lebih detail)
# ...

# 3. PASTIKAN TIPE DATA TETAP CATEGORY
df['bidang_utama'] = df['bidang_utama'].astype('category')

print("Pembersihan cerdas selesai!")
print(df[['nama_lomba', 'bidang_utama']].head(10)) # Cek hasilnya

Pembersihan cerdas selesai!
                         nama_lomba         bidang_utama
0             National DevFest 2025            Tech & IT
1     Festival Eco Summit Samarinda  Sosial & Lingkungan
2  Festival Health Challenge Malang            Kesehatan
3            National Ideathon 2024            Tech & IT
4                GELAR DEVFEST BALI            Tech & IT
5    Preneur Olympiad Denpasar 2024     Bisnis & Ekonomi
6             National DevFest 2025            Tech & IT
7  relawan challenge padang 2026     Sosial & Lingkungan
8    NATIONAL HEALTH CHALLENGE 2026            Kesehatan
9          Startup Fest Malang 2024     Bisnis & Ekonomi


In [9]:
print("\nMissing values di df:\n", df.isna().sum())


Missing values di df:
 nama_lomba         0
bidang_utama       0
kategori          90
tema              88
jumlah_peserta    85
hadiah             0
start_date        70
deadline_date     97
penyelenggara     70
dtype: int64


In [14]:
import pandas as pd

# Pastikan df adalah dataframe terakhirmu
# ==========================================
# 1. KATEGORI & TEMA (Konteks Dinamis)
# ==========================================
# Ubah sementara ke string agar gampang dimanipulasi, lalu kembalikan ke category
df['kategori'] = df.apply(
    lambda row: f"Umum - {row['bidang_utama']}" if pd.isna(row['kategori']) else str(row['kategori']), 
    axis=1
).astype('category')

df['tema'] = df.apply(
    lambda row: f"Umum - {row['bidang_utama']}" if pd.isna(row['tema']) else str(row['tema']), 
    axis=1
).astype('category')

# ==========================================
# 2. JUMLAH PESERTA (Grouped Median)
# ==========================================
# WAJIB: Pastikan jumlah_peserta menjadi numerik dulu agar tidak error saat transform median!
df['jumlah_peserta'] = pd.to_numeric(df['jumlah_peserta'], errors='coerce')

# Menghitung median peserta SPESIFIK untuk masing-masing bidang_utama (tambahkan observed=True)
df['jumlah_peserta'] = df['jumlah_peserta'].fillna(
    df.groupby('bidang_utama', observed=True)['jumlah_peserta'].transform('median')
)

# (Opsional) pastikan lagi jadi integer setelah di-median-kan
# Menggunakan 'Int64' (I besar) agar aman dari error jika masih ada sisa NaN tak terduga
df['jumlah_peserta'] = df['jumlah_peserta'].round().astype('Int64')
# ==========================================
# 3. PENYELENGGARA (Fallback Label)
# ==========================================
df['penyelenggara'] = df['penyelenggara'].fillna('Penyelenggara Independen').astype('object')

# ==========================================
# 4. TANGGAL (Forward Fill / Backward Fill)
# ==========================================
# ffill() akan menyalin tanggal dari baris sebelumnya yang tidak kosong
df['start_date'] = df['start_date'].ffill().bfill() 
df['deadline_date'] = df['deadline_date'].ffill().bfill()

# ==========================================
# CEK HASIL FINAL
# ==========================================
print("Missing values setelah sapu bersih:")
print(df.isnull().sum())

Missing values setelah sapu bersih:
nama_lomba        0
bidang_utama      0
kategori          0
tema              0
jumlah_peserta    0
hadiah            0
start_date        0
deadline_date     0
penyelenggara     0
dtype: int64


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1002 entries, 0 to 1099
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   nama_lomba      1002 non-null   object  
 1   bidang_utama    1002 non-null   category
 2   kategori        1002 non-null   category
 3   tema            1002 non-null   category
 4   jumlah_peserta  1002 non-null   Int64   
 5   hadiah          1002 non-null   object  
 6   start_date      1002 non-null   object  
 7   deadline_date   1002 non-null   object  
 8   penyelenggara   1002 non-null   object  
dtypes: Int64(1), category(3), object(5)
memory usage: 60.4+ KB


In [17]:
# ==========================================
# TYPE CASTING FINAL (UBAH TIPE DATA)
# ==========================================

# 1. Tipe Object (String / Teks)
df['nama_lomba'] = df['nama_lomba'].astype('object')
df['penyelenggara'] = df['penyelenggara'].astype('object')

# 2. Tipe Categorical
df['bidang_utama'] = df['bidang_utama'].astype('category')
df['kategori'] = df['kategori'].astype('category')
df['tema'] = df['tema'].astype('category')

# 3. Tipe Integer (int64)
# Peserta bisa langsung diubah karena desimal sudah dibulatkan
df['jumlah_peserta'] = df['jumlah_peserta'].astype('int64')

# Hadiah diamankan dulu dari string kosong (''), diubah ke numerik, 
# diisi median jika ada yang kosong, baru diubah ke int64
df['hadiah'] = pd.to_numeric(df['hadiah'], errors='coerce')
df['hadiah'] = df['hadiah'].fillna(df['hadiah'].median())
df['hadiah'] = df['hadiah'].astype('int64')

# 4. Tipe Datetime (datetime64[ns])
# errors='coerce' akan memastikan jika ada format yang gagal terbaca, tidak akan membuat program berhenti.
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['deadline_date'] = pd.to_datetime(df['deadline_date'], errors='coerce')

# ==========================================
# CEK HASIL AKHIR TIPE DATA
# ==========================================
print("Tipe data final setelah type casting:\n")
print(df.dtypes)

# Simpan dataset yang sudah sempurna ini
df.to_csv('dataset_lomba_siap_analisis.csv', index=False)
print("\n✅ Eksekusi Selesai! Data siap digunakan.")

Tipe data final setelah type casting:

nama_lomba                object
bidang_utama            category
kategori                category
tema                    category
jumlah_peserta             int64
hadiah                     int64
start_date        datetime64[ns]
deadline_date     datetime64[ns]
penyelenggara             object
dtype: object

✅ Eksekusi Selesai! Data siap digunakan.
